In [69]:
from dotenv import load_dotenv
import os
from langchain_snowflake import create_session_from_env
from langchain_snowflake import SnowflakeCortexSearchRetriever
import textwrap
from langchain_snowflake import ChatSnowflake
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from trulens.apps.langchain import TruChain
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core.run import Run, RunConfig
from datetime import datetime
from langchain_community.chat_models import ChatSnowflakeCortex
import pandas as pd
import time

In [2]:
load_dotenv()

True

In [5]:
session=create_session_from_env()

In [8]:
session

In [14]:
CORTEX_SEARCH_SERVICE="TEST.PUBLIC.SALES_CONVERSATION_SEARCH"

In [15]:
retriever=SnowflakeCortexSearchRetriever(
    session=session,
    service_name=CORTEX_SEARCH_SERVICE,
    database='TEST',
    schema='PUBLIC',
    k=3,  # number of documets to retrieve
    auto_format_for_rag=True,
    content_field="TRANSCRIPT_TEXT",  # extract content from this field
    join_separator="\n\n",
    fallback_to_page_content=True  # Fallback to page_content if metadata field is empty
)

In [16]:
docs=retriever.get_relevant_documents(query="""
    What happened in our last sales conversation with DataDriven?
                                      """)

In [17]:
docs

[Document(metadata={'@scores': {'text_match': 0.08659726, 'cosine_similarity': 0.5331961}, 'TRANSCRIPT_TEXT': "In-depth demo session with DataDriven Co's Analytics team and Business Intelligence managers. Showcase focused on advanced analytics capabilities, custom dashboard creation, and real-time data processing features. Team was particularly impressed with our machine learning integration and predictive analytics models. Competitor comparison requested specifically against Market Leader Z and Innovative Start-up X. Price point falls within their allocated budget range, but team expressed interest in multi-year commitment with corresponding discount structure. Technical questions centered around data warehouse integration and custom visualization capabilities. Action items: prepare detailed competitor feature comparison matrix and draft multi-year pricing proposals with various discount scenarios.", '_formatted_for_rag': True, '_original_page_content': '', '_content_field_used': 'TRA

In [18]:
for doc in docs:
    wrapped_text=textwrap.fill(doc.page_content, width=80)
    print(f"{wrapped_text}\n{'-'*120}")

In-depth demo session with DataDriven Co's Analytics team and Business
Intelligence managers. Showcase focused on advanced analytics capabilities,
custom dashboard creation, and real-time data processing features. Team was
particularly impressed with our machine learning integration and predictive
analytics models. Competitor comparison requested specifically against Market
Leader Z and Innovative Start-up X. Price point falls within their allocated
budget range, but team expressed interest in multi-year commitment with
corresponding discount structure. Technical questions centered around data
warehouse integration and custom visualization capabilities. Action items:
prepare detailed competitor feature comparison matrix and draft multi-year
pricing proposals with various discount scenarios.
------------------------------------------------------------------------------------------------------------------------
Initial discovery call with TechCorp Inc's IT Director and Solutions Architec

In [21]:
llm=ChatSnowflake(session=session, 
                  model="llama3.1-405b", 
                  temperature=0.1, 
                  max_tokens=1000)

In [22]:
llm.invoke(input="Who is Sachin Tendulkar?")

AIMessage(content='A legendary figure in the world of cricket!\n\nSachin Ramesh Tendulkar is a former Indian international cricketer widely regarded as one of the greatest batsmen in the history of the game. Born on April 24, 1973, in Mumbai, India, Tendulkar is often referred to as the "God of Cricket" by his fans.\n\nTendulkar\'s impressive career spanned over two decades, from 1989 to 2013. He played for the Indian national team in all formats of the game, including Test matches, One-Day Internationals (ODIs), and Twenty20 Internationals (T20Is). During his career, he set numerous records, many of which still stand today.\n\nSome of his notable achievements include:\n\n1. **Highest run-scorer in international cricket**: Tendulkar scored 100 international centuries (49 in ODIs and 51 in Tests) and accumulated 34,357 runs in international cricket.\n2. **First batsman to score 200 in ODIs**: Tendulkar achieved this feat in 2010, scoring 200 not out against South Africa.\n3. **Youngest 

In [24]:
ragPrompt=ChatPromptTemplate.from_template(template="""
    Answer the question based on the following context from Snowflake Cortex Search:
                                           
    Context:
    {context}
    
    Question:
    {question}
                                        
    Provide a comprehensive answer based on the retrieved context. If the context does not contain enough information, say so clearly                                       
    """)

In [25]:
ragChain={
    'question':RunnablePassthrough(),
    'context':retriever
} | ragPrompt| llm|StrOutputParser()

In [26]:
response=ragChain.invoke(input="What happened in our last sales conversation with DataDriven?")

In [27]:
print(response)

According to the retrieved context, our last sales conversation with DataDriven Co was an in-depth demo session with their Analytics team and Business Intelligence managers. Here's a summary of what happened:

* We showcased our advanced analytics capabilities, custom dashboard creation, and real-time data processing features, which impressed the team.
* They were particularly interested in our machine learning integration and predictive analytics models.
* The team requested a competitor comparison specifically against Market Leader Z and Innovative Start-up X.
* Our price point falls within their allocated budget range, but they expressed interest in a multi-year commitment with a corresponding discount structure.
* Technical questions centered around data warehouse integration and custom visualization capabilities.
* Action items from the conversation include preparing a detailed competitor feature comparison matrix and drafting multi-year pricing proposals with various discount sce

In [36]:
chatModel=ChatSnowflakeCortex(
    model='llama3.1-405b',
    cortex_function='complete',
    temperature=0.2,
    max_tokens=100,
    top_p=0.95,
    account=os.environ.get("SNOWFLAKE_ACCOUNT"),
    username=os.environ.get("SNOWFLAKE_USER"),
    password=os.environ.get("SNOWFLAKE_PASSWORD"),
    database=os.environ.get("SNOWFLAKE_DATABASE"),
    schema=os.environ.get("SNOWFLAKE_SCHEMA"),
    role=os.environ.get('SNOWFLAKE_ROLE'),
    warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE')
)

In [37]:
ragChain2={
    'question':RunnablePassthrough(),
    'context':retriever
} | ragPrompt| chatModel|StrOutputParser()

In [38]:
response=ragChain2.invoke(input="What happened in our last sales conversation with DataDriven?")

In [39]:
print(response)

Based on the retrieved context, our last sales conversation with DataDriven Co was an in-depth demo session with their Analytics team and Business Intelligence managers. The conversation focused on showcasing our advanced analytics capabilities, custom dashboard creation, and real-time data processing features. The team was particularly impressed with our machine learning integration and predictive analytics models.

The client requested a competitor comparison specifically against Market Leader Z and Innovative Start-up X. Our price point falls within their allocated budget range, but they expressed interest in a multi-year


In [42]:
tru_snowflake_connector=SnowflakeConnector(snowpark_session=session)

Running the TruLens dashboard requires providing a `password` to the `SnowflakeConnector`.


In [43]:
app_name="sales_assistance_rag"
app_version="cortex_search"

In [56]:
ragChain

{
  question: RunnablePassthrough(),
  context: SnowflakeCortexSearchRetriever(session=<snowflake.snowpark.session.Session object at 0x0000014DF7444E10>, database='TEST', schema='PUBLIC', service_name='TEST.PUBLIC.SALES_CONVERSATION_SEARCH', k=3)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n    Answer the question based on the following context from Snowflake Cortex Search:\n\n    Context:\n    {context}\n\n    Question:\n    {question}\n\n    Provide a comprehensive answer based on the retrieved context. If the context does not contain enough information, say so clearly                                       \n    '), additional_kwargs={})])
| ChatSnowflake(session=<snowflake.snowpark.session.Session object at 0x0000014DF7444E10>, model='llama3.1-405b', temperature=0.1, max_tokens=1

In [47]:
tru_rag=TruChain(
    app=ragChain,
    app_name=app_name,
    app_version=app_version, 
    connector=tru_snowflake_connector,
    main_method_name='invoke'
    )


instrumenting <class 'langchain_core.runnables.base.RunnableParallel'> for base <class 'langchain_core.runnables.base.RunnableParallel'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
instrumenting <class 'langchain_core.runnables.base.RunnableParallel'> for base <class 'langchain_core.runnables.base.RunnableSerializable[-Input, dict[str, Any]]'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
instrumenting <class 'langchain_core.runnables.base.RunnableParallel'> for base <class 'langchain_core.runnables.base.RunnableSerializable'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
instrumenting <class 'langchain_core.runnables.base.RunnableParallel'> for base <class 'langchain_core.load.serializable.Serializable'>
instrumenting <class 'langchain_core.runnables.passthrough.RunnablePassthrough'> for base <class 'langchain_core.runnables.passthrough.RunnablePassthro

In [49]:
queries=[
    "What happened in our last sales conversation with DataDriven?",
    "What is the status of the deal with DataDriven?",
    "What is the status of the deal with HealthTech?"
]

In [50]:
queriesDF=pd.DataFrame(data=queries,columns=['query'])
queriesDF

,query
0,What happened in our last sales conversation w...
1,What is the status of the deal with DataDriven?
2,What is the status of the deal with HealthTech?


In [76]:
run_name=f"experiment_1_{datetime.now().strftime('%Y%m%d%H%M%S')}"
run_name

'experiment_1_20251114233938'

In [77]:
run_config=RunConfig(
    run_name=run_name,
    dataset_name='sales_queries',
    source_type='DATAFRAME',
    dataset_spec={
            'input': 'query'
    }
)

In [78]:
run = tru_rag.add_run(run_config=run_config)

In [79]:
run.start(input_df=queriesDF)

In [80]:
while run.get_status()!='INVOCATION_COMPLETED':
    time.sleep(3)

computedValue=run.compute_metrics(metrics=[
    'answer_relevance',
    'context_relevance',
    'groundedness'
])

In [88]:
computedValue

'Metrics computation in progress.'